In [ ]:
# Imports
# TODO: change if necessary to accomodate change in data used
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from dataclasses import dataclass, asdict
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [ ]:
# Code to obtain activation of network
def get_activation(name):
    activations = {
        "relu": nn.ReLU,
        "gelu": nn.GELU,
        "silu": nn.SiLU,
        "tanh": nn.Tanh,
        "identity": nn.Identity,
    }
    name = name.lower()
    if name not in activations:
        raise ValueError(f"Unknown activation {name}")
    return activations[name]()

In [ ]:
# Expert metadata dataclass
@dataclass
class ExpertMetadata:
    expert_id: int
    family: str
    architecture: str
    depth: int
    width: int
    activation: str
    geometry: str = "euclidean"
    num_parameters: int = 0

In [ ]:
# Base expert
class BaseExpert(nn.Module):
    """
    Parent class of all experts
    Each expert has common input/output interface, exposes architectural metadata and
    can be analyzed by router's logging system
    TODO: Use this in a later step for the hyperbolic neural net subclass
    """

    def __init__(
        self,
        input_dim=256,
        output_dim=256,
        expert_id=-1,
        family="unknown",
        architecture="unknown",
        depth=0,
        width=0,
        activation="unknown",
        geometry="euclidean",
    ):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.metadata = ExpertMetadata(
            expert_id=expert_id,
            family=family,
            architecture=architecture,
            depth=depth,
            width=width,
            activation=activation,
            geometry=geometry,
        )

    def forward(self, x):
        raise NotImplementedError(
        )

    def update_parameter_count(self):
        """Update number of trainable and non-trainable parameters."""
        self.metadata.num_parameters = sum(
            p.numel() for p in self.parameters()
        )

    def get_metadata(self):
        """Return a copy of the expert's metadata as a dictionary."""
        self.update_parameter_count()
        return asdict(self.metadata)

In [ ]:
# MLPExpert
class MLPExpert(BaseExpert):
    """
    Subclass of BaseExpert, default activation is gelu
    """
    def __init__(
        self,
        input_dim=256,
        output_dim=256,
        hidden_dims=(512,),
        activation="gelu",
        residual=False,
        expert_id=-1,
        family="mlp",
        geometry="euclidean",
    ):
        hidden_dims = tuple(hidden_dims)
        # Determine metadata from the actual architecture.
        dims = [input_dim] + list(hidden_dims) + [output_dim]

        architecture = " -> ".join(map(str, dims))

        if residual:
            architecture += " + residual"

        super().__init__(
            input_dim=input_dim,
            output_dim=output_dim,
            expert_id=expert_id,
            family=family,
            architecture=architecture,
            depth=len(dims) - 1,
            width=max(hidden_dims, default=0),
            activation=activation,
            geometry=geometry,
        )
        self.hidden_dims = hidden_dims
        self.activation_name = activation
        self.residual = residual
        if residual and input_dim != output_dim:
            raise ValueError(
                f"res conns req input_dim == output_dim, input dim is {self.input_dim} and output dim is {self.output_dim}"
            )
        
        layers = []

        for i in range(len(dims) - 1):
            layers.append(
                nn.Linear(dims[i], dims[i + 1])
            )
            # Activation after all hidden layers
            if i < len(dims) - 2:
                layers.append(get_activation(activation))
        self.network = nn.Sequential(*layers)
        self.update_parameter_count()

    def forward(self, x):
        output = self.network(x)
        if self.residual:
            output = output + x
        return output

In [ ]:
# GeGLU, different because it's fundamentally different from the others
class GEGLUExpert(BaseExpert):
    """
    Gated GELU expert (also subclass of BaseExpert)
    """
    def __init__(
        self,
        input_dim=256,
        hidden_dim=512,
        output_dim=256,
        expert_id=-1,
        geometry="euclidean",
    ):
        super().__init__(
            input_dim=input_dim,
            output_dim=output_dim,
            expert_id=expert_id,
            family="gated",
            architecture=(
                f"{input_dim} -> GEGLU({hidden_dim}) -> {output_dim}"
            ),
            depth=2,
            width=hidden_dim,
            activation="gelu_gate",
            geometry=geometry,
        )
        self.hidden_dim = hidden_dim
        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim * 2,
        )
        self.output_projection = nn.Linear(
            hidden_dim,
            output_dim,
        )
        self.update_parameter_count()

    def forward(self, x):
        value, gate = self.input_projection(x).chunk(
            2, dim=-1
        )
        hidden = value * F.gelu(gate)
        return self.output_projection(hidden)

In [ ]:
# Output record of expert metadata
def log_routing_event(experts, probabilities, selected_id):
    expert = experts[selected_id]
    metadata = expert.get_metadata()

    record = {
        **metadata,
        "routing_probability": probabilities[
            selected_id
        ].detach().item(),
    }
    return record

In [ ]:
# Expert "factory"
'''Automating creation of experts (because I'm a lazy bum)'''
def make_expert(
    expert_id,
    family,
    input_dim=256,
    output_dim=256
):

    # Shallow and wide
    if family == "shallow_wide":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(1024,),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 1024 -> 256",
            model,
            depth=2,
            width=1024,
            activation="gelu"
        )

    # Shallow and medium width
    elif family == "shallow_medium":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256",
            model,
            depth=2,
            width=512,
            activation="gelu"
        )

    # Deep and narrow width
    elif family == "deep_narrow":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(256, 256, 256),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 256 -> 256 -> 256 -> 256",
            model,
            depth=5,
            width=256,
            activation="gelu"
        )

    # Very deep and 
    elif family == "very_deep":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(192, 192, 192, 192, 192),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 192 x 5 -> 256",
            model,
            depth=7,
            width=192,
            activation="gelu"
        )

    # ReLU
    elif family == "relu":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="relu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256",
            model,
            depth=2,
            width=512,
            activation="relu"
        )

    # GELU
    elif family == "gelu":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256",
            model,
            depth=2,
            width=512,
            activation="gelu"
        )

    # SiLU
    elif family == "silu":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="silu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256",
            model,
            depth=2,
            width=512,
            activation="silu"
        )

    # Tanh
    elif family == "tanh":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="tanh"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256",
            model,
            depth=2,
            width=512,
            activation="tanh"
        )

    # GeGLU
    elif family == "gated":

        model = GEGLUExpert(
            input_dim=input_dim,
            hidden_dim=512,
            output_dim=output_dim
        )

        return Expert(
            expert_id,
            family,
            "256 -> GEGLU(512) -> 256",
            model,
            depth=2,
            width=512,
            activation="GEGLU"
        )

    # Residual
    elif family == "residual":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(512,),
            activation="gelu",
            residual=True
        )

        return Expert(
            expert_id,
            family,
            "256 -> 512 -> 256 + residual",
            model,
            depth=2,
            width=512,
            activation="gelu"
        )

    # Bottlenecking
    elif family == "bottleneck":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(64,),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 64 -> 256",
            model,
            depth=2,
            width=64,
            activation="gelu"
        )

    # Expanding
    elif family == "expansion":

        model = MLPExpert(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_dims=(1024,),
            activation="gelu"
        )

        return Expert(
            expert_id,
            family,
            "256 -> 1024 -> 256",
            model,
            depth=2,
            width=1024,
            activation="gelu"
        )

    else:
        raise ValueError(
            f"{family} is wrong"
        )


In [ ]:
# "Free" labour

families = [
    "shallow_wide",
    "shallow_medium",
    "deep_narrow",
    "very_deep",
    "relu",
    "gelu",
    "silu",
    "tanh",
    "gated",
    "residual",
    "bottleneck",
    "expansion",
]

# Constructing dynamic decision graph
# TODO: change this to a dict so I can access them easier
experts = nn.ModuleList(
    [
        make_expert(
            expert_id=i,
            family=families[i % len(families)]
        )
        for i in range(50)
    ]
)

# Inspecting metadata
for expert in experts:
    print(
        expert.get_metadata()
    )

In [ ]:

# Device stuff
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using:", device)

In [ ]:
# Training
# Note that currently using MNIST dataset that's not complicated for classification
# TODO: use word data instead so change everything (I want to see if this can help capture semantics)

# Data handling (convert then normalize)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.1307,),
        (0.3081,)
    )
])

# Datasets
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)


In [ ]:
# Model
# TODO: changed format of architecture so change this too
model = DynamicDecisionGraph(
    input_dim=28 * 28,
    state_dim=128,
    hidden_dim=128,
    num_classes=10,
    num_experts=8,
    collection=collection,
    max_steps=6
).to(device)

# Adam optimizer selected for its in-built regularizationa and performance
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3 # Current tentative
)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training
epochs = 100 # Note to self: modify LR as epochs increase, don't run from start

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        images = images.view(
            images.size(0),
            -1
        )

        # Forward pass
        logits = model(images)
        # Loss
        loss = criterion(
            logits,
            labels
        )

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Stats
        total_loss += loss.item()
        predictions = logits.argmax(dim=1)
        correct += (
            predictions == labels
        ).sum().item()
        total += labels.size(0)

    # Overall stats
    accuracy = correct / total
    print(
        f"Epoch {epoch + 1}/{epochs} "
        f"| Loss: {total_loss / len(train_loader):.4f} "
        f"| Accuracy: {accuracy:.4f}"
    )